<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NER**

* Asignatura: Procesamiento del Lenguaje Natural
* Profesor: Dr. Rogelio Florencia Juárez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* 12 de abril de 2026




Aplica CRF para el Reconocimiento de Entidades Nombradas al conjunto de datos proporcionado.


**Instrucciones:**

1.	Carga el conjunto de datos y revisa su contenido.
2.	Preprocesa los datos para generar las frases tal como las vistas en las diapositivas 19 y 20 de la presentación:
[
 ('Interesante', 'ADJ', 'O'),
 ('será', 'VLfin', 'O'),
 ('conocer', 'VLinf', 'O'),
 ('Pedro', 'NP', 'B-PER'),
 ('Kumamoto', 'NC', 'E-PER')
]

Para lo anterior, considera el siguiente código y, en caso necesario, haz las modificaciones necesarias (no he probado este código):

```
def agrupar_oraciones(df):
    oraciones = []

    grouped = df.groupby("Sentence #")

    for _, grupo in grouped:
        sentence = [(w, p, t) for w, p, t in zip(grupo["Word"], grupo["Pos"], grupo["Tag"])]
        oraciones.append(sentence)

    return oraciones


sentences = agrupar_oraciones(df)
print(sentences[0])
```

3.	Genera el conjunto train con el 80% de las frases formadas y el 20% para el conjunto test.
4.	Aplica las métricas para evaluar el desempeño del modelo.
5.	Muestra la matriz de confusión.
6.	Identifica y menciona cuál fue la etiqueta NER que consideres fue en la que CRF tuvo el más bajo desempeño.


In [32]:
# Importa e instala las librerías necesarias
!pip install sklearn-crfsuite
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import CRF
from sklearn_crfsuite import metrics
from sklearn_crfsuite.utils import flatten
from sklearn.metrics import confusion_matrix

In [33]:
# 1. Carga el conjunto de datos y revisa su contenido

conjuntodatos = pd.read_csv(
    "/content/drive/MyDrive/ClassFiles/split1.mx-news.txt",
    sep="\t",
    encoding="utf-8"
)
conjuntodatos.head()

,Sentence #,Word,Pos,Tag
0,Sentence 1,Interesante,ADJ,O
1,Sentence 1,será,VLfin,O
2,Sentence 1,conocer,VLinf,O
3,Sentence 1,las,ART,O
4,Sentence 1,reflexiones,NC,O


In [35]:
# 2. Preprocesa los datos para generar las frases

def agrupar_oraciones(df):
    oraciones = []

    grouped = df.groupby("Sentence #")

    for _, grupo in grouped:
        sentence = [(w, p, t) for w, p, t in zip(grupo["Word"], grupo["Pos"], grupo["Tag"])]
        oraciones.append(sentence)

    return oraciones

sentences = agrupar_oraciones(conjuntodatos)
print(sentences[0])

[('Interesante', 'ADJ', 'O'), ('será', 'VLfin', 'O'), ('conocer', 'VLinf', 'O'), ('las', 'ART', 'O'), ('reflexiones', 'NC', 'O'), ('de', 'PREP', 'O'), ('Pedro', 'NP', 'B-PER'), ('Kumamoto', 'NC', 'E-PER'), ('en', 'PREP', 'O'), ('torno', 'NC', 'O'), ('a', 'PREP', 'O'), ('este', 'DM', 'O'), ('fracaso', 'NC', 'O'), (',', 'CM', 'O'), ('por', 'PREP', 'O'), ('el', 'ART', 'O'), ('momento', 'NC', 'O'), ('lo', 'ART', 'O'), ('único', 'NC', 'O'), ('que', 'CQUE', 'O'), ('ha', 'VHfin', 'O'), ('dicho', 'QU', 'O'), ('es', 'VSfin', 'O'), ('que', 'CQUE', 'O'), ('al', 'PAL', 'O'), ('interior', 'NC', 'O'), ('de', 'PREP', 'O'), ('su', 'PPO', 'O'), ('grupo', 'NC', 'O'), ('existen', 'VLfin', 'O'), ('dos', 'CARD', 'O'), ('planteamientos', 'NC', 'O'), (',', 'CM', 'O'), ('uno', 'CARD', 'O'), ('seguir', 'VLinf', 'O'), ('por', 'PREP', 'O'), ('la', 'ART', 'O'), ('ruta', 'NC', 'O'), ('independiente', 'ADJ', 'O'), (',', 'CM', 'O'), ('el', 'ART', 'O'), ('otro', 'QU', 'O'), ('destaca', 'VLfin', 'O'), ('las', 'ART', '

In [36]:
# 3. Genera el conjunto train con el 80% de las frases formadas y el 20% para el conjunto test.

train_sents, test_sents = train_test_split(sentences, test_size=0.2, random_state=42)

In [37]:
len(train_sents), len(test_sents)

(1036, 259)

In [38]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'word': word,
        'postag': postag,
    }

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, postag, label in sent]

In [39]:
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

In [41]:
print(X_train[0])
print(y_train[0])

[{'word': 'Consideró', 'postag': 'VLfin'}, {'word': 'que', 'postag': 'CQUE'}, {'word': 'un', 'postag': 'ART'}, {'word': 'incremento', 'postag': 'NC'}, {'word': 'de', 'postag': 'PREP'}, {'word': '350', 'postag': 'CARD'}, {'word': 'o', 'postag': 'CC'}, {'word': '400', 'postag': 'CARD'}, {'word': 'pesos', 'postag': 'NC'}, {'word': 'para', 'postag': 'PREP'}, {'word': 'una', 'postag': 'ART'}, {'word': 'beca', 'postag': 'NC'}, {'word': ',', 'postag': 'CM'}, {'word': 'aunque', 'postag': 'CSUBX'}, {'word': 'parece', 'postag': 'VLfin'}, {'word': 'poco', 'postag': 'QU'}, {'word': ',', 'postag': 'CM'}, {'word': 'multiplicado', 'postag': 'VLadj'}, {'word': 'por', 'postag': 'PREP'}, {'word': 'el', 'postag': 'ART'}, {'word': 'número', 'postag': 'NC'}, {'word': 'de', 'postag': 'PREP'}, {'word': 'becarios', 'postag': 'NC'}, {'word': 'es', 'postag': 'VSfin'}, {'word': 'una', 'postag': 'ART'}, {'word': 'cantidad', 'postag': 'NC'}, {'word': 'de', 'postag': 'PREP'}, {'word': 'varios', 'postag': 'QU'}, {'w

In [42]:
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

In [43]:
print(len(X_test), len(y_test))

259 259


In [44]:
crf = CRF(
    algorithm='lbfgs',
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, max_iterations=100)

In [45]:
# 4. Aplica las métricas para evaluar el desempeño del modelo.
y_pred = crf.predict(X_test)

print(y_pred[0])

print(metrics.flat_classification_report(y_test, y_pred))

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TIT', 'I-TIT', 'I-TIT', 'E-TIT', 'O', 'B-PER', 'I-PER', 'E-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'E-PER', 'O']
              precision    recall  f1-score   support

       B-ADD       0.86      0.55      0.67        11
       B-AGE       0.83      0.62      0.71         8
       B-DAT       0.89      0.83      0.86        92
       B-DOC       1.00      0.06      0.11        17
       B-EVT       1.00      0.24      0.38        17
       B-FAC       0.00      0.00      0.00        11
       B-GPE       0.90      0.58      0.70        33
       B-LOC       0.00      0.00      0.00         2
       B-MNY       1.00      0.67      0.80        12
       B-ORG       0.84      0.47      0.60        88
       B-PER       0.79      0.80      0.79       153
       B-PEX       1.00      0.50      0.67        14
       B-PRC       1.00      1.00      1.00        

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
# 5. Muestra la matriz de confusión.

y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)

labels = list(crf.classes_)

cm = confusion_matrix(y_test_flat, y_pred_flat, labels=labels)

print(cm)

[[7944    0    0 ...    0    0    0]
 [   4    8    0 ...    0    0    0]
 [   3    0   19 ...    0    0    0]
 ...
 [   0    0    0 ...    3    0    0]
 [   1    0    0 ...    0    0    0]
 [   0    0    0 ...    0    0    0]]


6. Identifica y menciona cuál fue la etiqueta NER que consideres fue en la que CRF tuvo el más bajo desempeño.

* Las etiquetas con peor desempeño son B-FAC, I-FAC, B-LOC, I-LOC, B-PRO e I-PRO y varias que empiezan con S, ya que tiene muy pocos datos de entrenamiento para estas clases y se asignaron a la clase "O" que domina el modelo, generando una alta exactitud general pero bajo desempeño en clases con menos frecuencia.